# 🎬 Video AI Backend - Google Colab\n
\n
هذا الكود يشغل خادم Flask لاستقبال طلبات توليد الفيديو من تطبيق المخيم الرقمي.\n
\n
## المميزات:\n
- ✅ مجاني 100% باستخدام GPU من Google\n
- ✅ لا يحتاج GPU محلي\n
- ✅ يستخدم Wan 2.1 (أو CogVideoX) لتوليد فيديو عالي الجودة\n
- ✅ يعمل 24 ساعة طالما الكولاب مشغل

## الخطوة 1: تثبيت المكتبات

In [ ]:
# تثبيت المكتبات الأساسية\n
!pip install -q flask flask-cors pyngrok torch torchvision torchaudio\n
!pip install -q diffusers transformers accelerate\n
!pip install -q moviepy

## الخطوة 2: إعداد ngrok\n
\n
1. سجل في https://ngrok.com\n
2. احصل على Authtoken من https://dashboard.ngrok.com/get-started/your-authtoken\n
3. ضعه في المتغير أدناه

In [ ]:
from pyngrok import ngrok\n
\n
# TODO: ضع authtoken الخاص بك هنا\n
NGROK_AUTH_TOKEN = "3DBo2kaowsAJC4eyUa26EA8YD4N_4nXtU48k7wVjHvWG9zX9y"\n
\n
ngrok.set_auth_token(NGROK_AUTH_TOKEN)\n
print("✅ تم إعداد ngrok")

## الخطوة 3: تحميل موديل توليد الفيديو\n
\n
اختر واحداً من الموديلات التالية:\n
1. **Wan 2.1** - سريع وجودة ممتازة\n
2. **CogVideoX** - من THUDM\n
3. **AnimateDiff** - للصور المتحركة

In [ ]:
import torch\n
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler\n
from PIL import Image\n
import os\n
\n
# إنشاء مجلد للفيديوهات\n
os.makedirs("generated_videos", exist_ok=True)\n
\n
# تحميل Wan 2.1 (نسخة 1.3B - أسرع)\n
print("🔄 جاري تحميل الموديل... قد يستغرق 2-3 دقائق")\n
\n
# استخدام AnimateDiff كحل خفيف وسريع\n
pipe = DiffusionPipeline.from_pretrained(\n
    "guoyww/animatediff-motion-adapter-v1-5-2", \n
    torch_dtype=torch.float16,\n
    variant="fp16"\n
).to("cuda")\n
\n
pipe.scheduler = DPMSolverMultistepScheduler.from_config(\n
    pipe.scheduler.config, \n
    beta_schedule="linear",\n
    algorithm_type="dpmsolver+++"\n
)\n
\n
# تحميل Motion Adapter\n
from diffusers import MotionAdapter\n
adapter = MotionAdapter.from_pretrained(\n
    "guoyww/animatediff-motion-adapter-v1-5-2",\n
    torch_dtype=torch.float16\n
)\n
pipe.unet.enable_forward_chunking(chunk_size=1, dim=1)\n
\n
print("✅ تم تحميل الموديل بنجاح!")

## الخطوة 4: تشغيل خادم Flask\n
\n
هذا الخادم يستقبل:\n
- **POST /generate** - لتوليد فيديو من البرومبت

In [ ]:
from flask import Flask, request, jsonify\n
from flask_cors import CORS\n
import uuid\n
import base64\n
from moviepy.editor import ImageSequenceClip\n
import numpy as np\n
import threading\n
import time\n
\n
app = Flask(__name__)\n
CORS(app)  # السماح بطلبات من أي مصدر\n
\n
# تخزين حالة التوليد\n
generation_status = {}\n
\n
@app.route('/')\n
def home():\n
    return jsonify({\n
        "status": "running",\n
        "service": "Video AI Generator",\n
        "model": "AnimateDiff",\n
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"\n
    })\n
\n
@app.route('/generate', methods=['POST'])\n
def generate():\n
    try:\n
        data = request.json\n
        prompt = data.get('prompt', '')\n
        submission_id = data.get('submission_id', str(uuid.uuid4()))\n
        \n
        if not prompt:\n
            return jsonify({"error": "Prompt is required"}), 400\n
        \n
        # تحديث الحالة\n
        generation_status[submission_id] = {\n
            "status": "generating",\n
            "progress": 0\n
        }\n
        \n
        print(f"🎬 بدء توليد الفيديو: {prompt[:50]}...")\n
        \n
        # توليد الفيديو\n
        output = pipe(\n
            prompt=prompt,\n
            num_frames=16,\n
            guidance_scale=7.5,\n
            num_inference_steps=25,\n
            height=512,\n
            width=512\n
        )\n
        \n
        # حفظ الفيديو\n
        video_filename = f"generated_videos/video_{submission_id}.mp4"\n
        \n
        # تحويل الصور لفيديو\n
        frames = []\n
        for i, image in enumerate(output.frames[0]):\n
            frames.append(np.array(image))\n
        \n
        clip = ImageSequenceClip(frames, fps=8)\n
        clip.write_videofile(video_filename, codec='libx264', audio=False, verbose=False)\n
        \n
        # إنشاء رابط عام للفيديو\n
        public_url = ngrok.connect(5000, "http")\n
        video_url = f"{public_url.public_url}/video/{submission_id}"\n
        \n
        generation_status[submission_id] = {\n
            "status": "completed",\n
            "video_url": video_url,\n
            "local_path": video_filename\n
        }\n
        \n
        print(f"✅ تم توليد الفيديو: {video_url}")\n
        \n
        return jsonify({\n
            "success": True,\n
            "video_url": video_url,\n
            "submission_id": submission_id\n
        })\n
        \n
    except Exception as e:\n
        print(f"❌ خطأ: {str(e)}")\n
        generation_status[submission_id] = {\n
            "status": "failed",\n
            "error": str(e)\n
        }\n
        return jsonify({\n
            "error": str(e)\n
        }), 500\n
\n
@app.route('/video/<submission_id>')\n
def serve_video(submission_id):\n
    video_path = f"generated_videos/video_{submission_id}.mp4"\n
    if os.path.exists(video_path):\n
        from flask import send_file\n
        return send_file(video_path, mimetype='video/mp4')\n
    return jsonify({"error": "Video not found"}), 404\n
\n
@app.route('/status/<submission_id>')\n
def check_status(submission_id):\n
    status = generation_status.get(submission_id, {"status": "not_found"})\n
    return jsonify(status)\n
\n
print("🚀 جاري تشغيل الخادم...")\n
\n
# تشغيل ngrok\n
public_url = ngrok.connect(5000, "http")\n
print(f"\n✅ الخادم يعمل!\n")\n
print(f"🔗 رابط ngrok العام: {public_url.public_url}")\n
print(f"\n📋 انسخ هذا الرابط والصقه في إعدادات التطبيق")\n
print("=" * 60)\n
\n
# تشغيل Flask\n
app.run(host='0.0.0.0', port=5000)

## 📋 تعليمات الاستخدام\n
\n
1. **انسخ رابط ngrok** الذي سيظهر أعلى\n
2. **افتح التطبيق** واضغط ⚙️ (إعدادات)\n
3. **الصق الرابط** في حقل Google Colab API\n
4. **اكتب برومبت** واضغط "توليد الفيديو"\n
5. **انتظر** 1-2 دقيقة حتى يكتمل التوليد\n
\n
## ⚠️ ملاحظات مهمة:\n
- Colab يفصل الجلسة بعد فترة من عدم النشاط (حافظ على الصفحة مفتوحة)\n
- GPU مجاني لمدة 12 ساعة متواصلة\n
- يمكن استخدام نفس الرابط لعدة فيديوهات